.. _notebooks-pickling:

# Pickling objects

Pickling an object means saving it in a binary format (usually a `PKL` file). Unpickling means converting the `PKL` back to a Python object. It can be really useful when some calculations take a lot of time and you do not want to recompute them every time.

.. warnign
   Do not use this option for long terme storage. Every time the object source code changes, the unpickled object will have undetermined behavior.

In [ ]:
from importlib import resources
import os
from pprint import pprint

from lightwin.config.config_manager import process_config
from lightwin.core.accelerator.accelerator import Accelerator
from lightwin.ui.workflow_setup import run_simulation
from lightwin.util.pickling import MyCloudPickler

## Manual pickling/unpickling

### Perform first calculation

We start with an example configuration.

In [ ]:
toml_filepath = resources.files("lightwin.data.ads") / "lightwin.toml"

toml_keys = {
    "files": "files",
    "beam": "beam",
    "beam_calculator": "generic_envelope1d",
    "wtf": "generic_wtf",
    "design_space": "fit_phi_s_design_space",
    "plots": "plots_minimal",
}
override = {
    "wtf": {"k": 5},
    "plots": {
        "kwargs": {"lw": 5},
        "emittance": False,
        "energy": False,
        "envelopes": False,
        "phase": False,
        "twiss": False,
    },
}

config = process_config(toml_filepath, toml_keys, override=override)
fault_scenarios = run_simulation(config)

### Pickle the Accelerator

In [ ]:
pickler = MyCloudPickler()
fault_scenario = fault_scenarios[0]

reference = fault_scenario.ref_acc
ref_pickle_path = reference.pickle(pickler, path="reference-accelerator.pkl")

fixed = fault_scenario.fix_acc
fix_pickle_path = fixed.pickle(pickler, path="fixed-accelerator.pkl")

del fault_scenario, reference, fixed

.. warning::
    SimulationOutput instances created by Cython do not seem to be pickable.

### Unpickle the Accelerator

In [ ]:
reference = Accelerator.from_pickle(pickler, ref_pickle_path)
fixed = Accelerator.from_pickle(pickler, fix_pickle_path)

We can quickly check that we recovered the results.

In [ ]:
ax = None
for accelerator in (reference, fixed):
    ax = accelerator.plot("v_cav_mv", marker="o", ax=ax)

In [ ]:
for path in (ref_pickle_path, fix_pickle_path):
    os.remove(path)
del reference, fixed

## Configure pickling/unpickling from the `TOML`

The `pickle_paths` entry in the `TOML` should associate every accelerator name with a `PKL` filepath.

.. code-block:: toml

    [files]
    dat_file = "ads.dat"
    project_folder = "lw_results/"

    [files.pickle_paths]
    reference = "reference.pkl"

    # Scenario 1: pre-computed solution (skips optimization)
    [files.pickle_paths.scenarios.000001]
    solution = "solution-000001.pkl"

    # Scenario 2: alternatives with custom names (optimization still runs, the pickled Accelerators will be appended)
    [files.pickle_paths.scenarios.000002.alternatives]
    "Conservative approach" = "design-conservative.pkl"
    "Aggressive tuning" = "design-aggressive.pkl"

    # Scenario 3: solution + alternatives
    [files.pickle_paths.scenarios.000003]
    solution = "solution-000003.pkl"

    [files.pickle_paths.scenarios.000003.alternatives]
    "Tweaked design" = "tweaked.pkl"
    "Experimental config" = "experimental.pkl"


- If the path does not exist, the simulation is performed normally and the associated :class:`.Accelerator` object is pickled.
- If the path exists, the :class:`.Accelerator` is unpickled and associated calculations are skipped.

.. warning::
   After every update/source code editing, you should remove your pickles in order to avoid hard-to-track down bugs.


### First simple scenario

Here, we define a pickle path for both the `Reference` :class:`.Accelerator` and the `Solution` of the first (and unique) :class:`.FaultScenario`.

.. note::
   `"Reference"` and `"Solution"` are reserved keywords for `Reference` and `Solution` :class:`.Accelerator`\s.

In [ ]:
pickle_paths = {
    "Reference": "reference.pkl",
    "scenarios": {
        "000001": {
            "Solution": "solution-000001.pkl"
        },
    }
}
override["files"] = {"pickle_paths": pickle_paths}
config = process_config(toml_filepath, toml_keys, override=override)

Both files `"reference.pkl"` and `"solution-000001.pkl"` do not exist. So a normal simulation will be run, and corresponding :class:`.Accelerator`\s will be pickled to this paths. 

In [ ]:
fault_scenarios = run_simulation(config)

Now that `"reference.pkl"` and `"solution-000001.pkl"` do exist, :class:`.Accelerator`\s are unpickled and simulations are skipped.

In [ ]:
fault_scenarios = run_simulation(config)